<a href="https://colab.research.google.com/github/Ravi-attada/Algorithms-and-Analysis/blob/main/Algorithms_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

minflow max cut

In [1]:
from collections import deque, defaultdict

class Graph:
    def __init__(self, vertices):
        self.V = vertices
        self.graph = defaultdict(lambda: defaultdict(int))

    def add_edge(self, u, v, capacity):
        self.graph[u][v] = capacity

    def bfs(self, s, t, parent):
        visited = [False] * self.V
        queue = deque([s])
        visited[s] = True

        while queue:
            u = queue.popleft()

            for v, cap in self.graph[u].items():
                if not visited[v] and cap > 0:
                    queue.append(v)
                    visited[v] = True
                    parent[v] = u
                    if v == t:
                        return True
        return False

    def edmonds_karp(self, source, sink):
        parent = [-1] * self.V
        max_flow = 0

        while self.bfs(source, sink, parent):
            path_flow = float("inf")
            v = sink
            while v != source:
                u = parent[v]
                path_flow = min(path_flow, self.graph[u][v])
                v = u

            max_flow += path_flow

            # update residual capacities
            v = sink
            while v != source:
                u = parent[v]
                self.graph[u][v] -= path_flow
                self.graph[v][u] += path_flow
                v = u

        return max_flow

    # Find the minimum cut edges
    def min_cut(self, source):
        visited = [False] * self.V
        queue = deque([source])
        visited[source] = True

        while queue:
            u = queue.popleft()
            for v, cap in self.graph[u].items():
                if cap > 0 and not visited[v]:
                    visited[v] = True
                    queue.append(v)

        cut_edges = []
        for u in range(self.V):
            if visited[u]:
                for v, cap in self.graph[u].items():
                    if cap == 0 and not visited[v]:  # saturated edge
                        cut_edges.append((u, v))

        return cut_edges


g = Graph(6)
g.add_edge(0, 1, 16)
g.add_edge(0, 2, 13)
g.add_edge(1, 2, 10)
g.add_edge(1, 3, 12)
g.add_edge(2, 1, 4)
g.add_edge(2, 4, 14)
g.add_edge(3, 2, 9)
g.add_edge(3, 5, 20)
g.add_edge(4, 3, 7)
g.add_edge(4, 5, 4)

source = 0
sink = 5

maxflow = g.edmonds_karp(source, sink)
print("Maximum Flow:", maxflow)

cut_edges = g.min_cut(source)
print("Min-Cut Edges:", cut_edges)


Maximum Flow: 23
Min-Cut Edges: [(1, 3), (4, 3), (4, 5)]


INTEROR POINT METHOD

In [9]:
# ============================================================
# Linear Program:
# minimize   3x1 + 2x2
# subject to:
#     x1 + x2 >= 4
#     2x1 + x2 <= 5
#     x1 >= 0, x2 >= 0
# ============================================================

def lp_solve():
    # Solve manually by checking all vertices
    vertices = []

    # Constraint lines:
    # C1: x1 + x2 = 4
    # C2: 2x1 + x2 = 5
    # Axes: x1 = 0, x2 = 0

    # 1. Intersection C1 & C2
    # x1 + x2 = 4
    # 2x1 + x2 = 5  --> subtract
    # x1 = 1, x2 = 3
    x1 = 1
    x2 = 3
    vertices.append((x1, x2))

    # 2. C1 & x1=0
    # x2 = 4
    vertices.append((0, 4))

    # 3. C2 & x1=0
    # x2 = 5
    vertices.append((0, 5))

    # 4. C2 & x2=0
    # 2x1 = 5 → x1 = 2.5
    x1 = 2.5
    x2 = 0
    vertices.append((x1, x2))

    # Compute objective value
    feasible = []
    for (a, b) in vertices:
        if (a + b >= 4) and (2*a + b <= 5) and a >= 0 and b >= 0:
            feasible.append((a, b, 3*a + 2*b))

    # Pick minimum objective
    best = min(feasible, key=lambda x: x[2])
    return best


# ASCII PLOT (simple text visualization)
def ascii_plot(x_opt, y_opt):
    print("\nFeasible Region (text plot 0–6):\n")
    for y in range(6, -1, -1):
        row = ""
        for x in range(0, 7):
            if x == int(round(x_opt)) and y == int(round(y_opt)):
                row += " *"   # optimum point
            elif x + y >= 4 and 2*x + y <= 5:
                row += " ."
            else:
                row += "  "
        print(row)
    print("\n* = optimal solution\n")


# ------------------------------
# Run solver
# ------------------------------
sol_x1, sol_x2, z = lp_solve()

print("Optimal solution (no packages):")
print(f"x1 = {sol_x1}")
print(f"x2 = {sol_x2}")
print(f"Objective value = {z}")

ascii_plot(sol_x1, sol_x2)


Optimal solution (no packages):
x1 = 0
x2 = 4
Objective value = 8

Feasible Region (text plot 0–6):

              
 .            
 *            
   .          
              
              
              

* = optimal solution

